In [1]:
# Install dependencies
%pip install torch
%pip install -U transformers==4.57.1 trl==0.25.1 datasets==4.4.1
!pip install bitsandbytes
!pip install peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5

In [2]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

30

In [3]:
# Authenticate with HuggingFace
from google.colab import userdata
from huggingface_hub import login

In [4]:
# Set your HF_TOKEN in Colab secrets (key icon in sidebar)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [5]:
# Generate the training dataset
!python generate_calendar_dataset.py

Generated 1360 examples
  Train: 1215
  Eval:  145
Saved to: calendar_training_data.jsonl


In [6]:
# Load Qwen2.5-1.5B
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="eager",
    dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# Load and format the training data
import json
from datasets import load_dataset

dataset = load_dataset("json", data_files="calendar_training_data.jsonl")["train"].shuffle(seed=42)

def apply_format_and_tokenize(sample):
    # 1. Generate full conversation text
    full_text = tokenizer.apply_chat_template(
        sample['messages'],
        tools=sample['tools'],
        tokenize=False,
        add_generation_prompt=False
    )

    # 2. Tokenize
    encoded = tokenizer(full_text, add_special_tokens=False)
    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # 3. Create labels array initialized to -100 (ignore everything by default)
    labels = [-100] * len(input_ids)

    # Qwen2.5 ChatML markers
    im_start = tokenizer.convert_tokens_to_ids("<|im_start|>")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")

    # Qwen encodes "assistant" differently depending on position,
    # but looking at the decoded string boundaries is most reliable.
    in_assistant_turn = False

    # 4. Walk the tokens and unmask only the assistant's outputs
    for i, token_id in enumerate(input_ids):
        # Look behind to see if we just passed `<|im_start|>assistant\n`
        # Qwen's template format is standard ChatML.
        if token_id == im_start:
            # Look ahead to verify it's the assistant
            # 77091 is usually 'assistant' in Qwen, but string matching is safer
            decoded_chunk = tokenizer.decode(input_ids[i:i+3])
            if "assistant" in decoded_chunk:
                in_assistant_turn = True
                continue # Don't compute loss on the prompt marker itself

        if in_assistant_turn:
            labels[i] = token_id

        if token_id == im_end and in_assistant_turn:
            in_assistant_turn = False

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "split": sample.get("metadata", "train"),
    }

processed_dataset = dataset.map(apply_format_and_tokenize)
train_dataset = processed_dataset.filter(lambda x: x['split'] == 'train')
eval_dataset = processed_dataset.filter(lambda x: x['split'] == 'eval')

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
max_tokens = max(len(x['input_ids']) for x in processed_dataset) + 100
print(f"Max token count: {max_tokens}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1360 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1360 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1360 [00:00<?, ? examples/s]

Train: 1215, Eval: 145
Max token count: 3356


In [8]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

# Switch to the standard Trainer to bypass TRL's strict formatting assumptions
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

output_dir = "/content/calendar-functiongemma"

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,     # <-- FORCE EVAL TO BATCH SIZE 1
    eval_accumulation_steps=1,        # <-- OFFLOAD EVAL MEMORY TO CPU
    gradient_accumulation_steps=32,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none"
)

base_model.config.pad_token_id = tokenizer.pad_token_id
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, padding=True)

base_model.enable_input_require_grads()

# 1. Define the LoRA adapter configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap your base model with the adapter
base_model = get_peft_model(base_model, peft_config)
base_model.print_trainable_parameters() # This will show you are only training ~1% of the model!

trainer = Trainer(
    model=base_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,0.431600,0.312108
100,0.208300,0.235623


Training complete!


In [9]:
# Push to HuggingFace Hub
from huggingface_hub import whoami

trained_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(output_dir)

username = whoami()['name']
hf_repo_id = f"{username}/qwen2.5-1.5b-calendar-agent"

trained_model.push_to_hub(hf_repo_id, create_repo=True, commit_message="Calendar agent fine-tune")
tokenizer.push_to_hub(hf_repo_id)
print(f"Uploaded to: https://huggingface.co/{hf_repo_id}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6wg8f6na/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

Uploaded to: https://huggingface.co/amoghghadge/functiongemma-270m-calendar-agent


In [1]:
# 1. Install Hugging Face dependencies for merging the LoRA adapter
!pip install -U torch transformers==4.57.1 peft accelerate

# 2. Uninstall TensorFlow to prevent conflicts with the Edge converter
!pip uninstall -y tensorflow

# 3. Install the AI Edge nightly builds for LiteRT conversion
!pip install ai-edge-torch-nightly --force-reinstall
!pip install ai-edge-litert-nightly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
     ━━━━━━━━━━━━━━━

In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# --- STEP 1: MERGE LORA WEIGHTS ---
print("Merging LoRA weights with base model...")
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Pull the adapter directly from your Hugging Face repo instead of the wiped local path
adapter_id = "amoghghadge/qwen2.5-1.5b-calendar-agent"
merged_dir = "/content/merged-qwen"

# Load the base model to CPU to prevent OOM errors during export
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="cpu",
    dtype=torch.bfloat16 # Fixed the deprecation warning
)

# Apply and merge the adapter directly from the Hub
peft_model = PeftModel.from_pretrained(base_model, adapter_id)
merged_model = peft_model.merge_and_unload()

# Save the full, merged PyTorch model locally for the LiteRT converter
os.makedirs(merged_dir, exist_ok=True)
merged_model.save_pretrained(merged_dir)

# Pull the tokenizer from your repo and save it with the merged model
tokenizer = AutoTokenizer.from_pretrained(adapter_id)
tokenizer.save_pretrained(merged_dir)
print("Merge complete!")


# --- STEP 2: CONVERT TO LITETLM ---
from ai_edge_torch.generative.examples.qwen import qwen
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache

# Metadata for Qwen2.5 ChatML format
llm_metadata = r"""start_token: {
    token_str: "<|im_start|>"
}
stop_tokens: {
    token_str: "<|im_end|>"
}
llm_model_type: {
    qwen: {}
}
"""

litertlm_output_dir = '/content/litertlm'
os.makedirs(litertlm_output_dir, exist_ok=True)

metadata_path = os.path.join(litertlm_output_dir, 'base_llm_metadata.textproto')
with open(metadata_path, 'w') as f:
    f.write(llm_metadata)

print("Building PyTorch model for export...")
# Build the model. If this throws an attribute error, check `dir(qwen)`
# as the exact method name fluctuates in the nightly builds (e.g., build_model_1_5b).
pytorch_model = qwen.build_1_5b_model(merged_dir)

export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

print("Converting to LiteRT-LM format...")
converter.convert_to_litert(
    pytorch_model,
    output_path=litertlm_output_dir,
    output_name_prefix="mobile-qwen",
    prefill_seq_len=256,
    kv_cache_max_len=1024,
    quantize="dynamic_int8",
    export_config=export_config,
    # Qwen uses tokenizer.json instead of a SentencePiece .model file
    tokenizer_model_path=os.path.join(merged_dir, 'tokenizer.json'),
    base_llm_metadata_path=metadata_path,
    output_format="litertlm",
)

print("Conversion complete!")

Merging LoRA weights with base model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Merge complete!


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.9.1, so it will not be used.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_xla2/distributed.py:106: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda` or `xpu`. Please specify it via the `devices` argument of `register_backend`.
  dist.Backend.register_backend("jax", ProcessGroupJax)
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 497, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 348, in initialize
    xla_client.register_custom_type_id_handler(
    ^^^^^^^^^^^^^^^^^

Building PyTorch model for export...
Converting to LiteRT-LM format...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/local/lib/python3.12/dist-packages/ai_edge_torch/_convert/signature.py:52: FutureWarning: `treespec.children_specs` is deprecated. Use `treespec.child(index)` to access a single child, or `treesp

Conversion complete!


In [4]:
from google.colab import drive
import os

# 1. Mount your Google Drive (This will pop up a permission prompt)
drive.mount('/content/drive')

# 2. Create a folder in your Drive to hold the model
drive_folder = '/content/drive/MyDrive/mobile-actions'
os.makedirs(drive_folder, exist_ok=True)

# 3. Copy the converted model from Colab's temporary storage to your Drive
# Note: Double check the exact filename output by the converter, but it should look like this:
!cp /content/litertlm/mobile-qwen_q8_ekv1024.litertlm /content/drive/MyDrive/mobile-actions/

print("Successfully backed up to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully backed up to Google Drive!
